# Gumbel-Softmax & XGES 快速测试

在小规模 ER 图上确认以下两个算法可正常运行（作为 benchmark 集成的前置验证）：

- **Gumbel-Softmax**：`nodag.nodag_gumbel_softmax.train_gumbel_sgd_dag`（项目内已有实现，带 DAG 化阈值搜索）
- **XGES**：`xges` PyPI 包（`pip install xges`；ANazaret/XGES 的官方纯 Python 实现）

In [1]:
# 1) 环境与导入
import os
import sys
import time

import numpy as np

repo_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..', '..'))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

from MEC import is_in_markov_equiv_class
from synthetic_dataset import SyntheticDataset

# ---- Gumbel-Softmax (已在项目内) ----
from nodag.nodag_gumbel_softmax import train_gumbel_sgd_dag
print('Gumbel-Softmax : OK')

# ---- XGES (pip install xges) ----
try:
    import xges
    print(f'XGES          : OK (v{xges.__version__})')
    HAS_XGES = True
except ImportError as e:
    print('XGES 不可用：', e)
    HAS_XGES = False

print('repo_root:', repo_root)

Gumbel-Softmax : OK
XGES          : OK (v0.1.6)
repo_root: C:\Users\super\DAG


In [2]:
# 2) 辅助函数：评估指标
def weight_to_binary_adj(W, threshold=0.05):
    G = (np.abs(W) > threshold).astype(int)
    np.fill_diagonal(G, 0)
    return G


def shd_score(G_true, G_est):
    G_true = np.asarray(G_true, dtype=int)
    G_est = np.asarray(G_est, dtype=int)
    d, dist = G_true.shape[0], 0
    for i in range(d):
        for j in range(i + 1, d):
            if G_true[i, j] != G_est[i, j] or G_true[j, i] != G_est[j, i]:
                dist += 1
    return dist


def summarize(alg, G_true, G_est, runtime):
    mec = int(is_in_markov_equiv_class(G_true, G_est))
    shd = shd_score(G_true, G_est)
    n_edges_true = int(G_true.sum())
    n_edges_est = int(G_est.sum())
    print(f'[{alg}] MEC={mec}  SHD={shd}  #edges est/true={n_edges_est}/{n_edges_true}  time={runtime:.3f}s')
    return {'alg': alg, 'mec': mec, 'shd': shd, 'runtime': runtime}

In [3]:
# 3) 生成一个小规模 ER 合成数据集
d = 10
n = 3000
seed = 42

dataset = SyntheticDataset(
    n=n, d=d,
    graph_type='ER',
    degree=2.0,
    noise_type='gaussian_nv',
    B_scale=1.0,
    seed=seed,
)
X = dataset.X
S = X.T @ X / X.shape[0]
G_true = weight_to_binary_adj(dataset.B, threshold=0.0)
print(f'X: {X.shape}  n_edges_true={int(G_true.sum())}')
print(G_true)

X: (3000, 10)  n_edges_true=15
[[0 0 0 0 0 0 0 0 0 0]
 [1 0 1 0 1 0 0 0 0 1]
 [0 0 0 0 0 0 0 0 0 0]
 [1 1 0 0 0 1 1 0 0 0]
 [0 0 0 0 0 0 0 0 0 0]
 [1 0 1 0 0 0 0 0 1 1]
 [0 0 1 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 1 0]
 [1 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0]]


In [4]:
# 4) Gumbel-Softmax 测试
# train_gumbel_sgd_dag 接收 Σ̂（样本协方差 S），通过阈值搜索保证输出 DAG

t0 = time.perf_counter()
B_final, G_gs, info_gs = train_gumbel_sgd_dag(
    Rhat_np=S,
    lam=0.5,
    delta=1e-6,
    max_steps=2000,
    lr=1e-2,
    tau_start=1.0,
    tau_end=0.2,
    history_every=500,
    seed=seed,
)
rt_gs = time.perf_counter() - t0

res_gs = summarize('gumbel_softmax', G_true, G_gs, rt_gs)
print('final_loss=', info_gs['final_loss'], ' thr=', info_gs['thr'])
print('G_est:')
print(G_gs)

[gumbel_softmax] MEC=0  SHD=16  #edges est/true=10/15  time=4.698s
final_loss= 102.29311167305875  thr= 0.6200000000000003
G_est:
[[0 0 0 0 0 0 0 1 0 0]
 [0 0 0 0 1 0 0 0 1 0]
 [0 0 0 0 0 0 0 0 0 0]
 [0 1 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 1 0]
 [1 0 0 1 1 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0]
 [1 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 1 0 0 0 0 0]]


In [5]:
# 5) XGES 测试
# XGES 返回 PDAG，pdag.get_dag_extension() 给出一个 MEC 内的 DAG

if not HAS_XGES:
    print('XGES 不可用，跳过。')
else:
    t0 = time.perf_counter()
    model = xges.XGES(alpha=2.0)
    # 注意：use_fast_numba=True 在当前环境（numba 0.59 + numpy 1.26）会触发类型不匹配，
    # 先用纯 Python 实现保证可跑通
    pdag = model.fit(X, verbose=0, use_fast_numba=False)
    rt_xges = time.perf_counter() - t0

    dag = pdag.get_dag_extension()
    G_xges = np.asarray(dag.to_adjacency_matrix(), dtype=int)
    np.fill_diagonal(G_xges, 0)

    res_xges = summarize('xges', G_true, G_xges, rt_xges)
    print('G_est (DAG extension):')
    print(G_xges)

[xges] MEC=1  SHD=1  #edges est/true=15/15  time=0.291s
G_est (DAG extension):
[[0 0 0 0 0 0 0 0 0 0]
 [1 0 1 0 1 0 0 0 0 1]
 [0 0 0 0 0 0 0 0 0 0]
 [1 1 0 0 0 0 1 0 0 0]
 [0 0 0 0 0 0 0 0 0 0]
 [1 0 1 1 0 0 0 0 1 1]
 [0 0 1 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 1 0]
 [1 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0]]


In [6]:
# 6) 汇总
print('结论：Gumbel-Softmax 与 XGES 均可正常运行。')
print('XGES 注意事项：use_fast_numba=False（numba 路径类型不匹配，待上游修复；纯 Python 仍快）。')

结论：Gumbel-Softmax 与 XGES 均可正常运行。
XGES 注意事项：use_fast_numba=False（numba 路径类型不匹配，待上游修复；纯 Python 仍快）。
